In [1319]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
import os
from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from skopt import BayesSearchCV
import warnings

## load_data

In [1320]:
def load_data(folder_path):
    dataframes = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            df = pd.read_csv(file_path)
            category = os.path.splitext(filename)[0]
            df['Category'] = category
            dataframes.append(df)
    if dataframes:
        merged_data = pd.concat(dataframes, ignore_index=True)
        merged_data['Date'] = pd.to_datetime(merged_data['Date'])
        merged_data = merged_data.sort_values(by='Date', ascending=True)
    else:
        print("No CSV files found in the specified folder.")
        return pd.DataFrame()

    return merged_data

## load_data2

In [1321]:
def load_data2(base_folder_path):
    dataframes = []
    
    for category in os.listdir(base_folder_path):
        category_path = os.path.join(base_folder_path, category)
        
        if os.path.isdir(category_path):
            for filename in os.listdir(category_path):
                if filename.endswith('.csv'):
                    file_path = os.path.join(category_path, filename)
                    df = pd.read_csv(file_path)
                    df['Category'] = category
                    province = os.path.splitext(filename)[0]
                    df['Province'] = province
                    price_columns = df.columns.difference(['Date', 'Category', 'Province'])
                    if not price_columns.empty:
                        df['Price'] = df[price_columns[0]]
                    else:
                        print(f"No price column found in {filename}. Skipping this file.")
                        continue
                    
                    dataframes.append(df)
    
    if dataframes:
        merged_data = pd.concat(dataframes, ignore_index=True)
        merged_data['Date'] = pd.to_datetime(merged_data['Date'])
    else:
        print("No CSV files found in the specified folder.")
        return pd.DataFrame()
    merged_data2 = merged_data.pivot_table(index=['Date', 'Category'], columns='Province', values='Price', aggfunc='first')

    merged_data2.reset_index(inplace=True)
    merged_data2['Category'] = merged_data2['Category'].replace({
    'bawang merah': 'Bawang Merah',
    'bawang putih': 'Bawang Putih Bonggol',
    'cabai merah': 'Cabai Merah Keriting',
    'cabai rawit': 'Cabai Rawit Merah',
    'daging ayam': 'Daging Ayam Ras',
    'daging sapi': 'Daging Sapi Murni',
    'gula': 'Gula Konsumsi',
    'telur ayam': 'Telur Ayam Ras',
    'tepung terigu': 'Tepung Terigu (Curah)',
})
    merged_data2 = merged_data2.sort_values(by='Date', ascending=True)
    return merged_data2


## melt

In [1322]:
def melt(df, value_name):
    melted = df.melt(id_vars=['Date', 'Category'], var_name='Province', value_name=value_name)
    return melted

## get_date

In [1323]:
def get_date(df, lag, target):
    df['Day'] = df.index.day
    df['day_of_week'] = df.index.dayofweek
    
    for i in range(1, lag + 1):
        df[f'lag_{i}'] = df.groupby('Category')[target].shift(i)
    
    df.fillna(method='bfill', inplace=True)

    return df 

## add_rolling_statistics

In [1324]:
def add_rolling_statistics(df, window_size, target, is_forecast, last_known_mean=None, last_known_std=None):
    if is_forecast:
        # For forecast, fill with last known values
        df[f'rolling_mean_{window_size}'] = last_known_mean
        df[f'rolling_std_{window_size}'] = last_known_std
    else:
        # Calculate rolling mean and std with min_periods=1 to avoid NaNs
        rolling_mean = df.groupby('Category')[target].rolling(window=window_size, min_periods=1).mean()
        rolling_std = df.groupby('Category')[target].rolling(window=window_size, min_periods=1).std()
        
        # Assign the rolling statistics back to the DataFrame
        df[f'rolling_mean_{window_size}'] = rolling_mean.reset_index(level=0, drop=True)
        df[f'rolling_std_{window_size}'] = rolling_std.reset_index(level=0, drop=True)

    # Fill NaN values using bfill
    df[f'rolling_mean_{window_size}'].fillna(method='bfill', inplace=True)
    df[f'rolling_std_{window_size}'].fillna(method='bfill', inplace=True)

    return df

## add_fourier_features

In [1325]:
def add_fourier_features(df, period):
    df[f'sin ({period})'] = np.sin(2 * np.pi * df.index.dayofyear / period)
    df[f'cos ({period})'] = np.cos(2 * np.pi * df.index.dayofyear / period)
    return df

## category_df

In [1326]:
def category_df(df, category):
    return df[df['Category'] == category]

## split_data

In [1327]:
def split_data(df, category, target):
    X = df.drop(columns=[target, 'Category'])
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    return X_train, X_test, y_train, y_test, X, y

## scale

In [1328]:
def scale(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return pd.DataFrame(X_train_scaled, columns=X_train.columns), pd.DataFrame(X_test_scaled, columns=X_test.columns)

## train_model

In [1329]:
def train_model(X_train, X_test, y_train, y_test, category):
    model = XGBRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    print(f'Mean Absolute Percentage Error for {category}: {mape}')
    return model

## make_df

In [1330]:
def make_df(df, target, last_known_value=None, last_known_mean=None, last_known_std=None):
    dates = pd.date_range(start='2024-10-01', end='2024-12-31', freq='D')
    prediction_set = pd.DataFrame(index=dates)
    
    # Ensure the last known category is correctly assigned
    prediction_set['Category'] = df['Category'].iloc[-1]  # Assuming this is a single value
    
    # Initialize the target column with the last known value or None
    prediction_set[target] = last_known_value if last_known_value is not None else None
    prediction_set = pd.concat([df, prediction_set])
    # Get date and rolling statistics
    prediction_set = get_date(prediction_set, 3, target, True, last_known_value)
    # prediction_set = add_rolling_statistics(prediction_set, 2, target, True, last_known_mean, last_known_std)
    
    return prediction_set

## forecast

In [1331]:
def forecast(df, model, target, category):
    df_forecast = df.copy()
    scaler = StandardScaler()
    scaler.fit(df.drop(columns=[target, 'Category']))
    
    for i in range(92):
        current_date = df_forecast.index[-1] + pd.Timedelta(days=1)
        new_row = pd.DataFrame(index=[current_date], columns=df.columns)
        new_row['Category'] = category
        new_row['Day'] = current_date.day
        new_row['day_of_week'] = current_date.dayofweek
        new_row = add_fourier_features(new_row, 365)
        new_row = add_fourier_features(new_row, 7)
        new_row = add_fourier_features(new_row, 30)
        
        for j in range(1, 4):
            new_row[f'lag_{j}'] = df_forecast[target].iloc[-j] if len(df_forecast) >= j else None
        
        if i == 0:
            new_row[target] = df_forecast[target].iloc[-1]
        else:
            new_row[target] = df_forecast[target].iloc[-1]
        
        X_new_row = new_row.drop(columns=[target, 'Category'])
        X_new_row_scaled = scaler.transform(X_new_row)
        y_forecast = model.predict(X_new_row_scaled)
        
        new_row[target] = y_forecast[0]
        
        # Update the lag features with the predicted value
        for j in range(1, 4):
            new_row[f'lag_{j}'] = df_forecast[target].iloc[-1] if j == 1 else new_row[f'lag_{j-1}']
        
        df_forecast = pd.concat([df_forecast, new_row])
    
    return df_forecast

## pipeline

In [1332]:
def pipeline(path, target, is_trend=False):
    result = []
    if is_trend:
        df = load_data2(path)
        df = melt(df, 'Trend')
    else:
        df = load_data(path)
    warnings.simplefilter(action='ignore', category=FutureWarning)
    warnings.simplefilter(action='ignore', category=Warning)
    feature_columns = ['Day', 'day_of_week'] #, 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_2', 'rolling_std_2']
    if target not in df.columns:
        raise KeyError(f"The target column '{target}' does not exist in the DataFrame.")
    
    if df[target].dtype == 'object':
        df[target] = df[target].replace({'\$':'', ',':'', '%':'', 'K':'000'}, regex=True)
        df[target] = df[target].apply(pd.to_numeric, errors='coerce')
    
    df = df.set_index('Date')   
    df = df[[target, 'Category']]
    
    
    df = get_date(df, 3, target)
    # df = add_rolling_statistics(df, 2, target, False)
    df = add_fourier_features(df, 365)
    df = add_fourier_features(df, 7)
    df = add_fourier_features(df, 30)

    df['Category'] = df['Category'].astype('category')
    categories = df['Category'].unique()
    
    for category in categories:
        temp_df = category_df(df, category)
        
        
        X_train, X_test, y_train, y_test, X, y = split_data(temp_df, category, target)
        X_train, X_test = scale(X_train, X_test)
        
        model = train_model(X_train, X_test, y_train, y_test, category)
        

        predictions = forecast(temp_df, model, target, category)
        result.append(predictions)
    
    result_df = pd.concat(result)
    return result_df

## Forecasting

In [1333]:
forecast_uang= pipeline(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\Mata Uang', 'Close', False)
forecast_uang.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')
forecast_commodity= pipeline(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\Global Commodity Price', 'Price', False)
forecast_commodity.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\commodity.csv')
forecast_trend= pipeline(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\Google Trend', 'Trend', True)
forecast_trend.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\trend.csv')

Mean Absolute Percentage Error for MYRUSD=X: 0.002391929900104161
Mean Absolute Percentage Error for SGDUSD=X: 0.0025619378286297376
Mean Absolute Percentage Error for USDIDR=X: 0.0034536994325295185
Mean Absolute Percentage Error for THBUSD=X: 0.004933891326365093
Mean Absolute Percentage Error for US Wheat Futures Historical Data: 0.022700646749866316
Mean Absolute Percentage Error for Palm Oil Futures Historical Data: 0.020211622841738518
Mean Absolute Percentage Error for Newcastle Coal Futures Historical Data: 0.021016904591468922
Mean Absolute Percentage Error for Natural Gas Futures Historical Data: 0.04082550437494077
Mean Absolute Percentage Error for Crude Oil WTI Futures Historical Data: 0.022857181511788872
Mean Absolute Percentage Error for US Sugar 11 Futures Historical Data: 0.015878395766998687
Mean Absolute Percentage Error for bawang: 1.5316311275743884e+16
Mean Absolute Percentage Error for Tepung Terigu (Curah): 1.3702620552680186e+16
Mean Absolute Percentage Error 

In [1279]:
forecast_uang.columns

Index(['Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'sin (365)', 'cos (365)', 'sin (7)', 'cos (7)', 'sin (30)', 'cos (30)'],
      dtype='object')

In [1280]:
forecast_uang.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')

In [1148]:
forecast_uang.columns

Index(['Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'sin (365)', 'cos (365)', 'sin (7)', 'cos (7)', 'sin (30)', 'cos (30)',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'sin (365)', 'cos (365)', 'sin (7)', 'cos (7)', 'sin (30)', 'cos (30)',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'sin (365)', 'cos (365)', 'sin (7)', 'cos (7)', 'sin (30)', 'cos (30)',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'sin (365)', 'cos (365)', 'sin (7)', 'cos (7)', 'sin (30)', 'cos (30)'],
      dtype='object')

In [1014]:
def pipeline(path, target):
    result = []
    df = load_data(path)
    warnings.simplefilter(action='ignore', category=FutureWarning)
    warnings.simplefilter(action='ignore', category=Warning)
    feature_columns = ['Day', 'day_of_week'] #, 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_2', 'rolling_std_2']
    if target not in df.columns:
        raise KeyError(f"The target column '{target}' does not exist in the DataFrame.")
    
    if df[target].dtype == 'object':
        df[target] = df[target].replace({'\$':'', ',':'', '%':'', 'K':'000'}, regex=True)
        df[target] = df[target].apply(pd.to_numeric, errors='coerce')
    
    df = df.set_index('Date')   
    df = df[[target, 'Category']]
    
    
    df = get_date(df, 3, target)
    # df = add_rolling_statistics(df, 2, target, False)

    df['Category'] = df['Category'].astype('category')
    temp_df = category_df(df, 'USDIDR=X')
    X_train, X_test, y_train, y_test, X, y = split_data(temp_df, 'USDIDR=X', target)
    X_train, X_test = scale(X_train, X_test)
        
    model = train_model(X_train, X_test, y_train, y_test, 'USDIDR=X')
        
    return temp_df, model

a, model = pipeline(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\Mata Uang', 'Close')

Mean Absolute Percentage Error for USDIDR=X: 0.003822120749715562
Trained model feature names: ['Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3']


In [ ]:
for i in range(92):
    current_date = a.index[-1] + pd.Timedelta(days=1) + pd.Timedelta(days=i)
    new_row = pd.DataFrame(index=[current_date], columns=a.columns)
    new_row['Category'] = 'USDIDR=X'
    for j in range(1, 4):
        new_row[f'lag_{j}'] = a['Close'].iloc[-j]
    # new_row = get_date(new_row, 3, 'Close')
    scaler = StandardScaler()
    X_new_row = scaler.fit_transform(new_row.drop(columns=['Close', 'Category']))
    y_forecast = model.predict(X_new_row)
    new_row['Close'] = y_forecast[0]
    a = pd.concat([a, new_row])
    print(new_row)


In [1015]:
for i in range(2):
    current_date = a.index[-1] + pd.Timedelta(days=1)
    new_row = pd.DataFrame(index=[current_date], columns=a.columns)
    new_row['Category'] = 'USDIDR=X'
    new_row['Day'] = new_row.index.day
    new_row['day_of_week'] = new_row.index.dayofweek
    for j in range(1, 4):
        new_row[f'lag_{j}'] = a['Close'].iloc[-j]
    # new_row = get_date(new_row, 3, 'Close')
        scaler = StandardScaler()
        X_new_row = scaler.fit_transform(new_row.drop(columns=['Close', 'Category']))
        y_forecast = model.predict(X_new_row)
        new_row['Close'] = y_forecast[0]
    a = pd.concat([a, new_row])
print(a[-2:])

                   Close  Category  Day  day_of_week         lag_1    lag_2  \
2024-10-01  15321.367563  USDIDR=X    1            1  15118.000000  15070.0   
2024-10-02  15321.367563  USDIDR=X    2            2  15321.367563  15118.0   

              lag_3  
2024-10-01  15201.5  
2024-10-02  15070.0  


In [ ]:
test.describe()

,MYRUSD=X,SGDUSD=X,THBUSD=X,USDIDR=X
count,92.000000,92.000000,92.000000,92.000000
mean,0.223496,0.748578,0.028486,16424.453125
std,0.000259,0.001308,0.000000,3.080189
min,0.221041,0.736186,0.028486,16421.261719
25%,0.223523,0.748700,0.028486,16424.166016
50%,0.223523,0.748700,0.028486,16424.166016
75%,0.223523,0.748700,0.028486,16424.166016
max,0.223523,0.749284,0.028486,16453.535156


In [ ]:
# test.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')